In [1]:
import os
import re
import fitz  # PyMuPDF

# ==========================================================
# CONFIGURATION
# ==========================================================
PDF_FILE = r"C:\Users\AD1HCZZ\Downloads\03.08.2026.pdf"
LABELS_PER_PAGE = 2

LOT_PATTERN = re.compile(r"([A-Z]+)(\d+)", re.IGNORECASE)

# ==========================================================
# FIND LOT
# ==========================================================
def find_lot(page, clip):

    words = page.get_text("words", clip=clip)

    for w in words:
        text = str(w[4]).strip().upper()

        if text == "LOT":
            lot_x = w[2]
            lot_y = (w[1] + w[3]) / 2

            candidates = []

            for c in words:
                candidate = str(c[4]).strip().upper()

                if LOT_PATTERN.fullmatch(candidate):

                    cx = c[0]
                    cy = (c[1] + c[3]) / 2

                    # same line as LOT
                    if abs(cy - lot_y) < 20 and cx > lot_x:

                        candidates.append(
                            (
                                abs(cx - lot_x),
                                candidate
                            )
                        )

            if candidates:
                candidates.sort()
                return candidates[0][1]

    # Backup Search
    for w in words:
        text = str(w[4]).strip().upper()

        if LOT_PATTERN.fullmatch(text):
            return text

    return None


# ==========================================================
# SORT KEY
# ==========================================================
def lot_key(lot):

    if lot is None:
        return ("ZZZ", 999999)

    m = LOT_PATTERN.fullmatch(lot)

    if m:
        return (
            m.group(1),
            int(m.group(2))
        )

    return ("ZZZ", 999999)


# ==========================================================
# MAIN FUNCTION
# ==========================================================
def sort_pdf_by_lot(src_path):

    out_path = os.path.splitext(src_path)[0] + "_sorted.pdf"

    src = fitz.open(src_path)

    labels = []

    print("\nDETECTED LABELS")
    print("-" * 60)

    for page_no, page in enumerate(src):

        width = page.rect.width
        height = page.rect.height

        slot_h = height / LABELS_PER_PAGE

        for slot in range(LABELS_PER_PAGE):

            clip = fitz.Rect(
                0,
                slot * slot_h,
                width,
                (slot + 1) * slot_h
            )

            lot = find_lot(page, clip)

            position = "TOP" if slot == 0 else "BOTTOM"

            print(
                f"Page {page_no + 1} "
                f"{position} -> {lot}"
            )

            labels.append(
                (
                    page_no,
                    clip,
                    lot
                )
            )

    print("\nTOTAL LABELS:", len(labels))

    # Sort labels
    labels.sort(key=lambda x: lot_key(x[2]))

    print("\nSORTED ORDER")
    print("-" * 60)

    for i, (_, _, lot) in enumerate(labels, start=1):
        print(i, lot)

    # Create output PDF
    out = fitz.open()

    width = src[0].rect.width
    height = src[0].rect.height

    slot_h = height / LABELS_PER_PAGE

    for i in range(0, len(labels), LABELS_PER_PAGE):

        new_page = out.new_page(
            width=width,
            height=height
        )

        batch = labels[i:i + LABELS_PER_PAGE]

        for slot, (page_no, clip, lot) in enumerate(batch):

            target = fitz.Rect(
                0,
                slot * slot_h,
                width,
                (slot + 1) * slot_h
            )

            new_page.show_pdf_page(
                target,
                src,
                page_no,
                clip=clip
            )

    out.save(
        out_path,
        garbage=3,
        deflate=True
    )

    out.close()
    src.close()

    print("\nOUTPUT FILE")
    print(out_path)

    return out_path


# ==========================================================
# RUN
# ==========================================================
if os.path.exists(PDF_FILE):
    sort_pdf_by_lot(PDF_FILE)
else:
    print("PDF NOT FOUND:")
    print(PDF_FILE)



DETECTED LABELS
------------------------------------------------------------
Page 1 TOP -> A17
Page 1 BOTTOM -> A18
Page 2 TOP -> A21
Page 2 BOTTOM -> A24
Page 3 TOP -> A32
Page 3 BOTTOM -> A25
Page 4 TOP -> A26
Page 4 BOTTOM -> A29
Page 5 TOP -> A36
Page 5 BOTTOM -> A30
Page 6 TOP -> A33
Page 6 BOTTOM -> A17
Page 7 TOP -> A18
Page 7 BOTTOM -> A21
Page 8 TOP -> A24
Page 8 BOTTOM -> A32
Page 9 TOP -> A25
Page 9 BOTTOM -> A26
Page 10 TOP -> A29
Page 10 BOTTOM -> A36
Page 11 TOP -> A30
Page 11 BOTTOM -> A33
Page 12 TOP -> A17
Page 12 BOTTOM -> A18
Page 13 TOP -> A21
Page 13 BOTTOM -> A24
Page 14 TOP -> A32
Page 14 BOTTOM -> A25
Page 15 TOP -> A26
Page 15 BOTTOM -> A29
Page 16 TOP -> A36
Page 16 BOTTOM -> A30
Page 17 TOP -> A33
Page 17 BOTTOM -> A17
Page 18 TOP -> A18
Page 18 BOTTOM -> A21
Page 19 TOP -> A24
Page 19 BOTTOM -> A32
Page 20 TOP -> A25
Page 20 BOTTOM -> A26
Page 21 TOP -> A29
Page 21 BOTTOM -> A36
Page 22 TOP -> A30
Page 22 BOTTOM -> A33
Page 23 TOP -> A17
Page 23 BOTTOM -> A